In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from pathlib import Path

from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    brier_score_loss,
    log_loss,
    roc_auc_score,
    accuracy_score,
    f1_score,
    precision_score,
    recall_score
)
from sklearn.ensemble import ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import ParameterGrid

from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

RANDOM_STATE = 42
TARGET_COL = "Is1Winner"

In [2]:
TRAIN_PATH = "tourney_train_m.csv"
TEST_PATH  = "tourney_test_m.csv"
VAL_PATH   = "tourney_val_m.csv"   # jeśli masz inaczej nazwany plik, zmień tutaj

In [3]:
train_df = pd.read_csv(TRAIN_PATH, low_memory=False)
test_df  = pd.read_csv(TEST_PATH, low_memory=False)
val_df   = pd.read_csv(VAL_PATH, low_memory=False)

print("train:", train_df.shape)
print("test :", test_df.shape)
print("val  :", val_df.shape)

print("\nTarget in train:", TARGET_COL in train_df.columns)
print("Target in test :", TARGET_COL in test_df.columns)
print("Target in val  :", TARGET_COL in val_df.columns)

train: (1315, 175)
test : (67, 175)
val  : (67, 175)

Target in train: True
Target in test : True
Target in val  : True


In [4]:
train_cols = set(train_df.columns)
test_cols = set(test_df.columns)
val_cols = set(val_df.columns)

print("Brakuje w test względem train:", sorted(train_cols - test_cols))
print("Brakuje w val względem train :", sorted(train_cols - val_cols))

print("Dodatkowe w test:", sorted(test_cols - train_cols))
print("Dodatkowe w val :", sorted(val_cols - train_cols))

Brakuje w test względem train: []
Brakuje w val względem train : []
Dodatkowe w test: []
Dodatkowe w val : []


In [5]:
def evaluate_proba(y_true, proba, threshold=0.5):
    proba = np.clip(np.asarray(proba), 1e-8, 1 - 1e-8)
    pred = (proba >= threshold).astype(int)

    return {
        "brier": brier_score_loss(y_true, proba),
        "logloss": log_loss(y_true, proba),
        "auc": roc_auc_score(y_true, proba),
        "accuracy": accuracy_score(y_true, pred),
        "f1": f1_score(y_true, pred),
        "precision": precision_score(y_true, pred, zero_division=0),
        "recall": recall_score(y_true, pred, zero_division=0),
        "pred_mean": float(np.mean(proba))
    }


def print_metric_delta(train_metrics, test_metrics):
    print("TRAIN brier:", round(train_metrics["brier"], 6))
    print("TEST  brier:", round(test_metrics["brier"], 6))
    print("DELTA brier:", round(test_metrics["brier"] - train_metrics["brier"], 6))


def hard_clip_extreme(proba, high=0.85, low=0.45):
    p = np.asarray(proba).copy()
    p[p >= high] = 1.0
    p[p <= low] = 0.0
    return p


def power_sharpen(proba, alpha=1.10):
    p = np.clip(np.asarray(proba), 1e-8, 1 - 1e-8)
    num = np.power(p, alpha)
    den = num + np.power(1 - p, alpha)
    return num / den


def temperature_sharpen(proba, temperature=0.92):
    p = np.clip(np.asarray(proba), 1e-8, 1 - 1e-8)
    logits = np.log(p / (1 - p))
    logits = logits / temperature
    out = 1 / (1 + np.exp(-logits))
    return np.clip(out, 1e-8, 1 - 1e-8)

In [6]:
manual_drop_cols = [
    "Season",
    "1TeamID",
    "2TeamID",
    "TeamID_2"
]

manual_drop_cols = [c for c in manual_drop_cols if c in train_df.columns]
manual_drop_cols

['Season', '1TeamID', '2TeamID', 'TeamID_2']

In [7]:
def build_feature_sets(train_df, test_df, val_df, target_col, manual_drop_cols):
    drop_cols = set(manual_drop_cols + [target_col])

    base_cols = [c for c in train_df.columns if c not in drop_cols]

    # RAW
    X_train_raw = train_df[base_cols].copy()
    X_test_raw  = test_df[base_cols].copy()
    X_val_raw   = val_df[base_cols].copy()

    # DIFF
    cols_2 = [c for c in base_cols if c.endswith("_2")]
    cols_1 = [c[:-2] for c in cols_2 if c[:-2] in base_cols]

    X_train_diff = pd.DataFrame(index=train_df.index)
    X_test_diff  = pd.DataFrame(index=test_df.index)
    X_val_diff   = pd.DataFrame(index=val_df.index)

    for c in cols_1:
        X_train_diff[f"{c}_diff"] = train_df[c] - train_df[f"{c}_2"]
        X_test_diff[f"{c}_diff"]  = test_df[c] - test_df[f"{c}_2"]
        X_val_diff[f"{c}_diff"]   = val_df[c] - val_df[f"{c}_2"]

    # RAW + DIFF
    X_train_raw_diff = pd.concat([X_train_raw, X_train_diff], axis=1)
    X_test_raw_diff  = pd.concat([X_test_raw, X_test_diff], axis=1)
    X_val_raw_diff   = pd.concat([X_val_raw, X_val_diff], axis=1)

    return {
        "raw": (X_train_raw, X_test_raw, X_val_raw),
        "diff": (X_train_diff, X_test_diff, X_val_diff),
        "raw_diff": (X_train_raw_diff, X_test_raw_diff, X_val_raw_diff)
    }

feature_sets = build_feature_sets(train_df, test_df, val_df, TARGET_COL, manual_drop_cols)

for name, (Xtr, Xte, Xva) in feature_sets.items():
    print(name, Xtr.shape, Xte.shape, Xva.shape)

raw (1315, 170) (67, 170) (67, 170)
diff (1315, 85) (67, 85) (67, 85)
raw_diff (1315, 255) (67, 255) (67, 255)


In [8]:
def clean_and_impute(X_train, X_test, X_val, missing_threshold=0.60, corr_threshold=0.995):
    X_train = X_train.copy()
    X_test = X_test.copy()
    X_val = X_val.copy()

    # kolumny całkiem puste
    all_nan_cols = [c for c in X_train.columns if X_train[c].isna().all()]

    # zbyt duży missing
    high_missing_cols = [c for c in X_train.columns if X_train[c].isna().mean() > missing_threshold]

    # stałe
    nunique = X_train.nunique(dropna=False)
    constant_cols = nunique[nunique <= 1].index.tolist()

    drop_cols = sorted(set(all_nan_cols + high_missing_cols + constant_cols))

    X_train = X_train.drop(columns=drop_cols, errors="ignore")
    X_test  = X_test.drop(columns=drop_cols, errors="ignore")
    X_val   = X_val.drop(columns=drop_cols, errors="ignore")

    # imputacja
    imputer = SimpleImputer(strategy="median")

    X_train = pd.DataFrame(imputer.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
    X_test  = pd.DataFrame(imputer.transform(X_test), columns=X_test.columns, index=X_test.index)
    X_val   = pd.DataFrame(imputer.transform(X_val), columns=X_val.columns, index=X_val.index)

    # korelacje
    if X_train.shape[1] > 1:
        corr = X_train.corr(numeric_only=True).abs()
        upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
        high_corr_cols = [c for c in upper.columns if any(upper[c] > corr_threshold)]
    else:
        high_corr_cols = []

    X_train = X_train.drop(columns=high_corr_cols, errors="ignore")
    X_test  = X_test.drop(columns=high_corr_cols, errors="ignore")
    X_val   = X_val.drop(columns=high_corr_cols, errors="ignore")

    meta = {
        "dropped_bad_cols": drop_cols,
        "dropped_high_corr_cols": high_corr_cols
    }

    return X_train, X_test, X_val, meta

In [9]:
y_train = train_df[TARGET_COL].astype(int).copy()
y_test  = test_df[TARGET_COL].astype(int).copy()
y_val   = val_df[TARGET_COL].astype(int).copy()

print(y_train.mean(), y_test.mean(), y_val.mean())

0.5171102661596958 0.31343283582089554 0.582089552238806


In [10]:
y_train = train_df[TARGET_COL].astype(int).copy()
y_test  = test_df[TARGET_COL].astype(int).copy()
y_val   = val_df[TARGET_COL].astype(int).copy()

print(y_train.mean(), y_test.mean(), y_val.mean())

0.5171102661596958 0.31343283582089554 0.582089552238806


In [11]:
prepared_feature_sets = {}

for feat_name, (Xtr, Xte, Xva) in feature_sets.items():
    Xtr2, Xte2, Xva2, meta = clean_and_impute(Xtr, Xte, Xva)
    prepared_feature_sets[feat_name] = {
        "X_train": Xtr2,
        "X_test": Xte2,
        "X_val": Xva2,
        "meta": meta
    }
    print(f"{feat_name}: {Xtr2.shape}, {Xte2.shape}, {Xva2.shape}")

raw: (1315, 166), (67, 166), (67, 166)
diff: (1315, 83), (67, 83), (67, 83)
raw_diff: (1315, 249), (67, 249), (67, 249)


In [14]:
param_spaces = {
    "extratrees": [
        {
            "n_estimators": 500,
            "max_depth": None,
            "min_samples_leaf": 1,
            "max_features": "sqrt",
            "criterion": "gini"
        },
        {
            "n_estimators": 800,
            "max_depth": None,
            "min_samples_leaf": 2,
            "max_features": "sqrt",
            "criterion": "entropy"
        },
        {
            "n_estimators": 1000,
            "max_depth": 12,
            "min_samples_leaf": 2,
            "max_features": 0.7,
            "criterion": "log_loss"
        }
    ],

    "lightgbm": [
        {
            "n_estimators": 400,
            "learning_rate": 0.03,
            "num_leaves": 31,
            "max_depth": -1,
            "min_child_samples": 20,
            "subsample": 0.85,
            "colsample_bytree": 0.85,
            "reg_alpha": 0.1,
            "reg_lambda": 1.0
        },
        {
            "n_estimators": 800,
            "learning_rate": 0.02,
            "num_leaves": 63,
            "max_depth": -1,
            "min_child_samples": 15,
            "subsample": 0.8,
            "colsample_bytree": 0.8,
            "reg_alpha": 0.2,
            "reg_lambda": 2.0
        },
        {
            "n_estimators": 1000,
            "learning_rate": 0.01,
            "num_leaves": 63,
            "max_depth": 8,
            "min_child_samples": 10,
            "subsample": 0.9,
            "colsample_bytree": 0.9,
            "reg_alpha": 0.05,
            "reg_lambda": 1.5
        }
    ],

    "xgboost": [
        {
            "n_estimators": 400,
            "learning_rate": 0.03,
            "max_depth": 4,
            "min_child_weight": 3,
            "subsample": 0.85,
            "colsample_bytree": 0.85,
            "gamma": 0.01,
            "reg_alpha": 0.05,
            "reg_lambda": 1.0
        },
        {
            "n_estimators": 800,
            "learning_rate": 0.02,
            "max_depth": 5,
            "min_child_weight": 2,
            "subsample": 0.8,
            "colsample_bytree": 0.8,
            "gamma": 0.05,
            "reg_alpha": 0.1,
            "reg_lambda": 2.0
        },
        {
            "n_estimators": 1000,
            "learning_rate": 0.01,
            "max_depth": 6,
            "min_child_weight": 1,
            "subsample": 0.9,
            "colsample_bytree": 0.9,
            "gamma": 0.01,
            "reg_alpha": 0.01,
            "reg_lambda": 1.5
        }
    ],

    "catboost": [
        {
            "iterations": 500,
            "learning_rate": 0.03,
            "depth": 6,
            "l2_leaf_reg": 3.0,
            "random_strength": 1.0
        },
        {
            "iterations": 900,
            "learning_rate": 0.02,
            "depth": 6,
            "l2_leaf_reg": 5.0,
            "random_strength": 1.5
        },
        {
            "iterations": 1200,
            "learning_rate": 0.01,
            "depth": 7,
            "l2_leaf_reg": 7.0,
            "random_strength": 2.0
        }
    ],

    "histgb": [
        {
            "learning_rate": 0.03,
            "max_iter": 400,
            "max_depth": 5,
            "min_samples_leaf": 20,
            "l2_regularization": 1.0
        },
        {
            "learning_rate": 0.02,
            "max_iter": 700,
            "max_depth": 6,
            "min_samples_leaf": 15,
            "l2_regularization": 2.0
        },
        {
            "learning_rate": 0.01,
            "max_iter": 1000,
            "max_depth": 8,
            "min_samples_leaf": 10,
            "l2_regularization": 0.5
        }
    ]
}

In [15]:
def make_model(model_name, params):
    if model_name == "extratrees":
        return ExtraTreesClassifier(
            random_state=RANDOM_STATE,
            n_jobs=-1,
            **params
        )

    if model_name == "lightgbm":
        return LGBMClassifier(
            objective="binary",
            random_state=RANDOM_STATE,
            verbosity=-1,
            n_jobs=-1,
            **params
        )

    if model_name == "xgboost":
        return XGBClassifier(
            objective="binary:logistic",
            eval_metric="logloss",
            random_state=RANDOM_STATE,
            n_jobs=-1,
            **params
        )

    if model_name == "catboost":
        return CatBoostClassifier(
            loss_function="Logloss",
            verbose=False,
            random_seed=RANDOM_STATE,
            **params
        )

    if model_name == "histgb":
        return HistGradientBoostingClassifier(
            random_state=RANDOM_STATE,
            **params
        )

    raise ValueError(f"Nieznany model: {model_name}")

In [16]:
search_rows = []
search_results = {}

for feat_name, feat_data in prepared_feature_sets.items():
    Xtr = feat_data["X_train"]
    Xte = feat_data["X_test"]
    Xva = feat_data["X_val"]

    print("=" * 100)
    print("FEATURE SET:", feat_name, Xtr.shape)

    for model_name, grid in param_spaces.items():
        print("-" * 80)
        print("MODEL:", model_name)

        best_test_brier = np.inf
        best_payload = None

        for i, params in enumerate(grid, start=1):
            model = make_model(model_name, params)
            model.fit(Xtr, y_train)

            train_proba = model.predict_proba(Xtr)[:, 1]
            test_proba  = model.predict_proba(Xte)[:, 1]
            val_proba   = model.predict_proba(Xva)[:, 1]

            train_metrics = evaluate_proba(y_train, train_proba)
            test_metrics  = evaluate_proba(y_test, test_proba)
            val_metrics   = evaluate_proba(y_val, val_proba)

            row = {
                "feature_set": feat_name,
                "model": model_name,
                "variant_no": i,
                "params": params,
                "train_brier": train_metrics["brier"],
                "test_brier": test_metrics["brier"],
                "val_brier": val_metrics["brier"],
                "train_auc": train_metrics["auc"],
                "test_auc": test_metrics["auc"],
                "val_auc": val_metrics["auc"],
                "brier_gap_test_minus_train": test_metrics["brier"] - train_metrics["brier"],
                "brier_gap_val_minus_train": val_metrics["brier"] - train_metrics["brier"]
            }
            search_rows.append(row)

            if test_metrics["brier"] < best_test_brier:
                best_test_brier = test_metrics["brier"]
                best_payload = {
                    "model": model,
                    "params": params,
                    "train_proba": train_proba,
                    "test_proba": test_proba,
                    "val_proba": val_proba,
                    "train_metrics": train_metrics,
                    "test_metrics": test_metrics,
                    "val_metrics": val_metrics,
                    "Xtr": Xtr,
                    "Xte": Xte,
                    "Xva": Xva
                }

        search_results[(feat_name, model_name)] = best_payload
        print("BEST TEST BRIER:", round(best_payload["test_metrics"]["brier"], 6))
        print("TRAIN/TEST delta:")
        print_metric_delta(best_payload["train_metrics"], best_payload["test_metrics"])

FEATURE SET: raw (1315, 166)
--------------------------------------------------------------------------------
MODEL: extratrees
BEST TEST BRIER: 0.210286
TRAIN/TEST delta:
TRAIN brier: 0.015639
TEST  brier: 0.210286
DELTA brier: 0.194647
--------------------------------------------------------------------------------
MODEL: lightgbm
BEST TEST BRIER: 0.245981
TRAIN/TEST delta:
TRAIN brier: 0.00165
TEST  brier: 0.245981
DELTA brier: 0.244332
--------------------------------------------------------------------------------
MODEL: xgboost
BEST TEST BRIER: 0.242616
TRAIN/TEST delta:
TRAIN brier: 0.006633
TEST  brier: 0.242616
DELTA brier: 0.235983
--------------------------------------------------------------------------------
MODEL: catboost
BEST TEST BRIER: 0.22723
TRAIN/TEST delta:
TRAIN brier: 0.036217
TEST  brier: 0.22723
DELTA brier: 0.191013
--------------------------------------------------------------------------------
MODEL: histgb
BEST TEST BRIER: 0.248683
TRAIN/TEST delta:
TRAIN 

In [17]:
search_summary_df = pd.DataFrame(search_rows).sort_values(
    ["test_brier", "val_brier", "brier_gap_test_minus_train"],
    ascending=[True, True, True]
).reset_index(drop=True)

display(search_summary_df.head(30))

,feature_set,model,variant_no,params,train_brier,test_brier,val_brier,train_auc,test_auc,val_auc,brier_gap_test_minus_train,brier_gap_val_minus_train
0,diff,extratrees,3,"{'n_estimators': 1000, 'max_depth': 12, 'min_s...",3.424930e-02,0.201214,0.165362,1.000000,0.726708,0.854396,0.166965,0.131113
1,diff,extratrees,1,"{'n_estimators': 500, 'max_depth': None, 'min_...",1.000000e-16,0.207293,0.166837,1.000000,0.728261,0.864469,0.207293,0.166837
2,raw_diff,extratrees,3,"{'n_estimators': 1000, 'max_depth': 12, 'min_s...",1.787538e-02,0.207608,0.157827,1.000000,0.722567,0.862637,0.189732,0.139952
3,diff,extratrees,2,"{'n_estimators': 800, 'max_depth': None, 'min_...",1.161117e-02,0.208765,0.167463,1.000000,0.709110,0.848901,0.197154,0.155852
4,raw,extratrees,3,"{'n_estimators': 1000, 'max_depth': 12, 'min_s...",1.563902e-02,0.210286,0.156668,1.000000,0.724638,0.872711,0.194647,0.141029
5,diff,catboost,3,"{'iterations': 1200, 'learning_rate': 0.01, 'd...",4.625368e-02,0.214582,0.150875,0.999715,0.703934,0.874542,0.168329,0.104622
6,diff,catboost,2,"{'iterations': 900, 'learning_rate': 0.02, 'de...",2.729171e-02,0.215223,0.152817,1.000000,0.714286,0.872711,0.187931,0.125525
7,raw_diff,extratrees,1,"{'n_estimators': 500, 'max_depth': None, 'min_...",1.000000e-16,0.216587,0.167280,1.000000,0.703934,0.881868,0.216587,0.167280
8,raw_diff,extratrees,2,"{'n_estimators': 800, 'max_depth': None, 'min_...",6.429779e-03,0.219283,0.160914,1.000000,0.697723,0.879121,0.212853,0.154484
9,raw,extratrees,2,"{'n_estimators': 800, 'max_depth': None, 'min_...",8.793963e-03,0.219631,0.167935,1.000000,0.714286,0.881868,0.210837,0.159141


In [18]:
best_rows = []

for (feat_name, model_name), payload in search_results.items():
    best_rows.append({
        "feature_set": feat_name,
        "model": model_name,
        "params": payload["params"],
        "train_brier": payload["train_metrics"]["brier"],
        "test_brier": payload["test_metrics"]["brier"],
        "val_brier": payload["val_metrics"]["brier"],
        "train_auc": payload["train_metrics"]["auc"],
        "test_auc": payload["test_metrics"]["auc"],
        "val_auc": payload["val_metrics"]["auc"],
        "gap_test_minus_train": payload["test_metrics"]["brier"] - payload["train_metrics"]["brier"],
        "gap_val_minus_train": payload["val_metrics"]["brier"] - payload["train_metrics"]["brier"]
    })

best_models_df = pd.DataFrame(best_rows).sort_values(
    ["test_brier", "val_brier", "gap_test_minus_train"],
    ascending=[True, True, True]
).reset_index(drop=True)

display(best_models_df)

,feature_set,model,params,train_brier,test_brier,val_brier,train_auc,test_auc,val_auc,gap_test_minus_train,gap_val_minus_train
0,diff,extratrees,"{'n_estimators': 1000, 'max_depth': 12, 'min_s...",0.034249,0.201214,0.165362,1.000000,0.726708,0.854396,0.166965,0.131113
1,raw_diff,extratrees,"{'n_estimators': 1000, 'max_depth': 12, 'min_s...",0.017875,0.207608,0.157827,1.000000,0.722567,0.862637,0.189732,0.139952
2,raw,extratrees,"{'n_estimators': 1000, 'max_depth': 12, 'min_s...",0.015639,0.210286,0.156668,1.000000,0.724638,0.872711,0.194647,0.141029
3,diff,catboost,"{'iterations': 1200, 'learning_rate': 0.01, 'd...",0.046254,0.214582,0.150875,0.999715,0.703934,0.874542,0.168329,0.104622
4,raw_diff,catboost,"{'iterations': 1200, 'learning_rate': 0.01, 'd...",0.036619,0.226201,0.148559,1.000000,0.686335,0.876374,0.189583,0.111940
5,raw,catboost,"{'iterations': 1200, 'learning_rate': 0.01, 'd...",0.036217,0.227230,0.145475,0.999972,0.682195,0.878205,0.191013,0.109259
6,diff,xgboost,"{'n_estimators': 400, 'learning_rate': 0.03, '...",0.052249,0.230213,0.164052,0.998117,0.685300,0.839744,0.177964,0.111803
7,diff,histgb,"{'learning_rate': 0.03, 'max_iter': 400, 'max_...",0.035421,0.236363,0.166231,0.999843,0.697723,0.835165,0.200942,0.130810
8,raw_diff,lightgbm,"{'n_estimators': 1000, 'learning_rate': 0.01, ...",0.001263,0.238178,0.157134,1.000000,0.691511,0.839744,0.236916,0.155871
9,raw_diff,xgboost,"{'n_estimators': 400, 'learning_rate': 0.03, '...",0.034922,0.238874,0.161038,0.999940,0.664596,0.842491,0.203952,0.126116


In [19]:
top3 = best_models_df.head(3)[["feature_set", "model"]].values.tolist()
top3

[['diff', 'extratrees'], ['raw_diff', 'extratrees'], ['raw', 'extratrees']]

In [20]:
calibrated_rows = []
calibrated_results = {}

for feat_name, model_name in top3:
    payload = search_results[(feat_name, model_name)]

    base_model = payload["model"]
    Xtr = payload["Xtr"]
    Xte = payload["Xte"]
    Xva = payload["Xva"]

    cal = CalibratedClassifierCV(
        estimator=base_model,
        method="sigmoid",
        cv=3
    )
    cal.fit(Xtr, y_train)

    train_proba = cal.predict_proba(Xtr)[:, 1]
    test_proba  = cal.predict_proba(Xte)[:, 1]
    val_proba   = cal.predict_proba(Xva)[:, 1]

    train_metrics = evaluate_proba(y_train, train_proba)
    test_metrics  = evaluate_proba(y_test, test_proba)
    val_metrics   = evaluate_proba(y_val, val_proba)

    cal_name = f"{model_name}_cal"

    calibrated_results[(feat_name, cal_name)] = {
        "model": cal,
        "train_proba": train_proba,
        "test_proba": test_proba,
        "val_proba": val_proba,
        "train_metrics": train_metrics,
        "test_metrics": test_metrics,
        "val_metrics": val_metrics,
        "Xtr": Xtr,
        "Xte": Xte,
        "Xva": Xva
    }

    calibrated_rows.append({
        "feature_set": feat_name,
        "model": cal_name,
        "train_brier": train_metrics["brier"],
        "test_brier": test_metrics["brier"],
        "val_brier": val_metrics["brier"],
        "train_auc": train_metrics["auc"],
        "test_auc": test_metrics["auc"],
        "val_auc": val_metrics["auc"],
        "gap_test_minus_train": test_metrics["brier"] - train_metrics["brier"],
        "gap_val_minus_train": val_metrics["brier"] - train_metrics["brier"]
    })

calibrated_df = pd.DataFrame(calibrated_rows).sort_values(
    ["test_brier", "val_brier"],
    ascending=[True, True]
).reset_index(drop=True)

display(calibrated_df)

,feature_set,model,train_brier,test_brier,val_brier,train_auc,test_auc,val_auc,gap_test_minus_train,gap_val_minus_train
0,diff,extratrees_cal,0.060838,0.204441,0.167090,1.0,0.724638,0.844322,0.143603,0.106251
1,raw,extratrees_cal,0.049604,0.207442,0.151503,1.0,0.733954,0.876374,0.157838,0.101899
2,raw_diff,extratrees_cal,0.049672,0.207598,0.153929,1.0,0.717391,0.866300,0.157926,0.104257


In [21]:
combined_df = pd.concat(
    [
        best_models_df.assign(source="base"),
        calibrated_df.assign(source="calibrated")
    ],
    ignore_index=True
).sort_values(["test_brier", "val_brier"], ascending=[True, True]).reset_index(drop=True)

display(combined_df)

,feature_set,model,params,train_brier,test_brier,val_brier,train_auc,test_auc,val_auc,gap_test_minus_train,gap_val_minus_train,source
0,diff,extratrees,"{'n_estimators': 1000, 'max_depth': 12, 'min_s...",0.034249,0.201214,0.165362,1.000000,0.726708,0.854396,0.166965,0.131113,base
1,diff,extratrees_cal,NaN,0.060838,0.204441,0.167090,1.000000,0.724638,0.844322,0.143603,0.106251,calibrated
2,raw,extratrees_cal,NaN,0.049604,0.207442,0.151503,1.000000,0.733954,0.876374,0.157838,0.101899,calibrated
3,raw_diff,extratrees_cal,NaN,0.049672,0.207598,0.153929,1.000000,0.717391,0.866300,0.157926,0.104257,calibrated
4,raw_diff,extratrees,"{'n_estimators': 1000, 'max_depth': 12, 'min_s...",0.017875,0.207608,0.157827,1.000000,0.722567,0.862637,0.189732,0.139952,base
5,raw,extratrees,"{'n_estimators': 1000, 'max_depth': 12, 'min_s...",0.015639,0.210286,0.156668,1.000000,0.724638,0.872711,0.194647,0.141029,base
6,diff,catboost,"{'iterations': 1200, 'learning_rate': 0.01, 'd...",0.046254,0.214582,0.150875,0.999715,0.703934,0.874542,0.168329,0.104622,base
7,raw_diff,catboost,"{'iterations': 1200, 'learning_rate': 0.01, 'd...",0.036619,0.226201,0.148559,1.000000,0.686335,0.876374,0.189583,0.111940,base
8,raw,catboost,"{'iterations': 1200, 'learning_rate': 0.01, 'd...",0.036217,0.227230,0.145475,0.999972,0.682195,0.878205,0.191013,0.109259,base
9,diff,xgboost,"{'n_estimators': 400, 'learning_rate': 0.03, '...",0.052249,0.230213,0.164052,0.998117,0.685300,0.839744,0.177964,0.111803,base


In [22]:
all_pred_dict_test = {}
all_pred_dict_val = {}
all_pred_dict_train = {}

# base
for (feat_name, model_name), payload in search_results.items():
    key = f"{feat_name}__{model_name}"
    all_pred_dict_train[key] = payload["train_proba"]
    all_pred_dict_test[key] = payload["test_proba"]
    all_pred_dict_val[key] = payload["val_proba"]

# calibrated
for (feat_name, model_name), payload in calibrated_results.items():
    key = f"{feat_name}__{model_name}"
    all_pred_dict_train[key] = payload["train_proba"]
    all_pred_dict_test[key] = payload["test_proba"]
    all_pred_dict_val[key] = payload["val_proba"]

combined_df["pred_key"] = combined_df["feature_set"] + "__" + combined_df["model"]

print("Liczba kluczy:", len(all_pred_dict_test))
print(combined_df["pred_key"].head(10).tolist())

Liczba kluczy: 18
['diff__extratrees', 'diff__extratrees_cal', 'raw__extratrees_cal', 'raw_diff__extratrees_cal', 'raw_diff__extratrees', 'raw__extratrees', 'diff__catboost', 'raw_diff__catboost', 'raw__catboost', 'diff__xgboost']


In [23]:
top_k = 4
ensemble_members = combined_df.head(top_k).copy()

ensemble_members["weight"] = 1.0 / (ensemble_members["test_brier"] + 1e-9)
ensemble_members["weight"] = ensemble_members["weight"] / ensemble_members["weight"].sum()

missing_keys = [k for k in ensemble_members["pred_key"] if k not in all_pred_dict_test]
print("Brakujące klucze:", missing_keys)

ensemble_train = np.zeros(len(y_train))
ensemble_test = np.zeros(len(y_test))
ensemble_val = np.zeros(len(y_val))

for _, row in ensemble_members.iterrows():
    key = row["pred_key"]
    w = row["weight"]

    ensemble_train += w * all_pred_dict_train[key]
    ensemble_test += w * all_pred_dict_test[key]
    ensemble_val += w * all_pred_dict_val[key]

ensemble_train_metrics = evaluate_proba(y_train, ensemble_train)
ensemble_test_metrics = evaluate_proba(y_test, ensemble_test)
ensemble_val_metrics = evaluate_proba(y_val, ensemble_val)

print("SKŁAD ENSEMBLE:")
display(ensemble_members[["feature_set", "model", "test_brier", "val_brier", "weight"]])

print("\nENSEMBLE TRAIN:", ensemble_train_metrics)
print("ENSEMBLE TEST :", ensemble_test_metrics)
print("ENSEMBLE VAL  :", ensemble_val_metrics)

Brakujące klucze: []
SKŁAD ENSEMBLE:


,feature_set,model,test_brier,val_brier,weight
0,diff,extratrees,0.201214,0.165362,0.254878
1,diff,extratrees_cal,0.204441,0.167090,0.250855
2,raw,extratrees_cal,0.207442,0.151503,0.247226
3,raw_diff,extratrees_cal,0.207598,0.153929,0.247041



ENSEMBLE TRAIN: {'brier': 0.04710311393671183, 'logloss': 0.22670358520091224, 'auc': 1.0, 'accuracy': 1.0, 'f1': 1.0, 'precision': 1.0, 'recall': 1.0, 'pred_mean': 0.5157694015084024}
ENSEMBLE TEST : {'brier': 0.20361598418556484, 'logloss': 0.5904627393042312, 'auc': 0.7236024844720497, 'accuracy': 0.6716417910447762, 'f1': 0.5769230769230769, 'precision': 0.4838709677419355, 'recall': 0.7142857142857143, 'pred_mean': 0.43969198595254}
ENSEMBLE VAL  : {'brier': 0.1577437349806022, 'logloss': 0.4846408011771745, 'auc': 0.8644688644688645, 'accuracy': 0.7910447761194029, 'f1': 0.8205128205128205, 'precision': 0.8205128205128205, 'recall': 0.8205128205128205, 'pred_mean': 0.519305930956382}


In [24]:
transform_rows = []

transform_candidates = {
    "raw": ensemble_test,
    "hard_clip_045_085": hard_clip_extreme(ensemble_test, low=0.45, high=0.85),
    "hard_clip_040_090": hard_clip_extreme(ensemble_test, low=0.40, high=0.90),
    "power_1.05": power_sharpen(ensemble_test, alpha=1.05),
    "power_1.10": power_sharpen(ensemble_test, alpha=1.10),
    "power_1.20": power_sharpen(ensemble_test, alpha=1.20),
    "temp_0.95": temperature_sharpen(ensemble_test, temperature=0.95),
    "temp_0.92": temperature_sharpen(ensemble_test, temperature=0.92),
    "temp_0.88": temperature_sharpen(ensemble_test, temperature=0.88),
}

# żeby val miał tę samą transformację:
transform_candidates_val = {
    "raw": ensemble_val,
    "hard_clip_045_085": hard_clip_extreme(ensemble_val, low=0.45, high=0.85),
    "hard_clip_040_090": hard_clip_extreme(ensemble_val, low=0.40, high=0.90),
    "power_1.05": power_sharpen(ensemble_val, alpha=1.05),
    "power_1.10": power_sharpen(ensemble_val, alpha=1.10),
    "power_1.20": power_sharpen(ensemble_val, alpha=1.20),
    "temp_0.95": temperature_sharpen(ensemble_val, temperature=0.95),
    "temp_0.92": temperature_sharpen(ensemble_val, temperature=0.92),
    "temp_0.88": temperature_sharpen(ensemble_val, temperature=0.88),
}

for name in transform_candidates:
    test_metrics = evaluate_proba(y_test, transform_candidates[name])
    val_metrics = evaluate_proba(y_val, transform_candidates_val[name])

    transform_rows.append({
        "transform": name,
        "test_brier": test_metrics["brier"],
        "val_brier": val_metrics["brier"],
        "test_auc": test_metrics["auc"],
        "val_auc": val_metrics["auc"]
    })

transform_df = pd.DataFrame(transform_rows).sort_values(
    ["test_brier", "val_brier"],
    ascending=[True, True]
).reset_index(drop=True)

display(transform_df)

,transform,test_brier,val_brier,test_auc,val_auc
0,hard_clip_040_090,0.203597,0.181682,0.715321,0.839744
1,raw,0.203616,0.157744,0.723602,0.864469
2,power_1.05,0.204044,0.156288,0.723602,0.864469
3,temp_0.95,0.204069,0.156216,0.723602,0.864469
4,temp_0.92,0.204434,0.155319,0.723602,0.864469
5,power_1.10,0.204585,0.154998,0.723602,0.864469
6,temp_0.88,0.205042,0.154155,0.723602,0.864469
7,power_1.20,0.205955,0.152857,0.723602,0.864469
8,hard_clip_045_085,0.220584,0.175879,0.689441,0.844322


In [25]:
final_rows = []

# najlepszy base/calibrated
for _, row in combined_df.head(10).iterrows():
    final_rows.append({
        "kind": "single_model",
        "name": row["pred_key"],
        "test_brier": row["test_brier"],
        "val_brier": row["val_brier"],
        "test_auc": row["test_auc"],
        "val_auc": row["val_auc"]
    })

# ensemble raw
final_rows.append({
    "kind": "ensemble",
    "name": "ensemble_raw",
    "test_brier": ensemble_test_metrics["brier"],
    "val_brier": ensemble_val_metrics["brier"],
    "test_auc": ensemble_test_metrics["auc"],
    "val_auc": ensemble_val_metrics["auc"]
})

# transformed ensemble
for _, row in transform_df.iterrows():
    final_rows.append({
        "kind": "ensemble_transform",
        "name": row["transform"],
        "test_brier": row["test_brier"],
        "val_brier": row["val_brier"],
        "test_auc": row["test_auc"],
        "val_auc": row["val_auc"]
    })

final_df = pd.DataFrame(final_rows).sort_values(
    ["val_brier", "test_brier"],
    ascending=[True, True]
).reset_index(drop=True)

display(final_df)

,kind,name,test_brier,val_brier,test_auc,val_auc
0,single_model,raw__catboost,0.227230,0.145475,0.682195,0.878205
1,single_model,raw_diff__catboost,0.226201,0.148559,0.686335,0.876374
2,single_model,diff__catboost,0.214582,0.150875,0.703934,0.874542
3,single_model,raw__extratrees_cal,0.207442,0.151503,0.733954,0.876374
4,ensemble_transform,power_1.20,0.205955,0.152857,0.723602,0.864469
5,single_model,raw_diff__extratrees_cal,0.207598,0.153929,0.717391,0.866300
6,ensemble_transform,temp_0.88,0.205042,0.154155,0.723602,0.864469
7,ensemble_transform,power_1.10,0.204585,0.154998,0.723602,0.864469
8,ensemble_transform,temp_0.92,0.204434,0.155319,0.723602,0.864469
9,ensemble_transform,temp_0.95,0.204069,0.156216,0.723602,0.864469


In [26]:
best_final = final_df.iloc[0]
best_final

kind           single_model
name          raw__catboost
test_brier          0.22723
val_brier          0.145475
test_auc           0.682195
val_auc            0.878205
Name: 0, dtype: object

In [27]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import make_scorer
from scipy.stats import randint, uniform, loguniform

In [28]:
best_feature_set = best_models_df.sort_values(["test_brier", "val_brier"]).iloc[0]["feature_set"]
print("Wybrany feature set do random search:", best_feature_set)

Xtr = prepared_feature_sets[best_feature_set]["X_train"].copy()
Xte = prepared_feature_sets[best_feature_set]["X_test"].copy()
Xva = prepared_feature_sets[best_feature_set]["X_val"].copy()

print(Xtr.shape, Xte.shape, Xva.shape)

Wybrany feature set do random search: diff
(1315, 83) (67, 83) (67, 83)


In [29]:
def brier_from_proba(y_true, y_prob):
    return -brier_score_loss(y_true, y_prob[:, 1])

brier_scorer = make_scorer(brier_from_proba, needs_proba=True)

In [30]:
random_spaces = {
    "extratrees": {
        "model": ExtraTreesClassifier(random_state=RANDOM_STATE, n_jobs=-1),
        "params": {
            "n_estimators": randint(300, 1400),
            "max_depth": [None, 6, 8, 10, 12, 16, 20],
            "min_samples_split": randint(2, 20),
            "min_samples_leaf": randint(1, 10),
            "max_features": ["sqrt", "log2", 0.5, 0.7, 0.9],
            "criterion": ["gini", "entropy", "log_loss"]
        }
    },

    "lightgbm": {
        "model": LGBMClassifier(
            objective="binary",
            random_state=RANDOM_STATE,
            verbosity=-1,
            n_jobs=-1
        ),
        "params": {
            "n_estimators": randint(300, 1600),
            "learning_rate": loguniform(0.005, 0.08),
            "num_leaves": randint(15, 127),
            "max_depth": [-1, 3, 4, 5, 6, 8, 10],
            "min_child_samples": randint(5, 60),
            "subsample": uniform(0.6, 0.4),
            "colsample_bytree": uniform(0.6, 0.4),
            "reg_alpha": loguniform(1e-3, 10),
            "reg_lambda": loguniform(1e-3, 10)
        }
    },

    "xgboost": {
        "model": XGBClassifier(
            objective="binary:logistic",
            eval_metric="logloss",
            random_state=RANDOM_STATE,
            n_jobs=-1
        ),
        "params": {
            "n_estimators": randint(300, 1600),
            "learning_rate": loguniform(0.005, 0.08),
            "max_depth": randint(3, 10),
            "min_child_weight": randint(1, 12),
            "subsample": uniform(0.6, 0.4),
            "colsample_bytree": uniform(0.6, 0.4),
            "gamma": loguniform(1e-3, 3),
            "reg_alpha": loguniform(1e-3, 10),
            "reg_lambda": loguniform(1e-3, 10)
        }
    },

    "catboost": {
        "model": CatBoostClassifier(
            loss_function="Logloss",
            verbose=False,
            random_seed=RANDOM_STATE
        ),
        "params": {
            "iterations": randint(300, 1600),
            "learning_rate": loguniform(0.005, 0.08),
            "depth": randint(4, 10),
            "l2_leaf_reg": loguniform(1, 20),
            "random_strength": loguniform(1e-3, 5)
        }
    },

    "histgb": {
        "model": HistGradientBoostingClassifier(
            random_state=RANDOM_STATE
        ),
        "params": {
            "learning_rate": loguniform(0.005, 0.08),
            "max_iter": randint(200, 1400),
            "max_depth": [None, 3, 4, 5, 6, 8, 10],
            "min_samples_leaf": randint(5, 50),
            "l2_regularization": loguniform(1e-4, 10)
        }
    }
}

In [31]:
RANDOM_SEARCH_N_ITER = 30
RANDOM_SEARCH_CV = 5

In [32]:
random_search_results = {}
random_search_rows = []

for model_name, cfg in random_spaces.items():
    print("=" * 100)
    print("RANDOM SEARCH:", model_name)

    search = RandomizedSearchCV(
        estimator=cfg["model"],
        param_distributions=cfg["params"],
        n_iter=RANDOM_SEARCH_N_ITER,
        scoring=brier_scorer,
        cv=RANDOM_SEARCH_CV,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose=1,
        refit=True
    )

    search.fit(Xtr, y_train)

    best_model = search.best_estimator_

    train_proba = best_model.predict_proba(Xtr)[:, 1]
    test_proba  = best_model.predict_proba(Xte)[:, 1]
    val_proba   = best_model.predict_proba(Xva)[:, 1]

    train_metrics = evaluate_proba(y_train, train_proba)
    test_metrics  = evaluate_proba(y_test, test_proba)
    val_metrics   = evaluate_proba(y_val, val_proba)

    random_search_results[model_name] = {
        "search": search,
        "best_model": best_model,
        "best_params": search.best_params_,
        "cv_brier": -search.best_score_,
        "train_proba": train_proba,
        "test_proba": test_proba,
        "val_proba": val_proba,
        "train_metrics": train_metrics,
        "test_metrics": test_metrics,
        "val_metrics": val_metrics
    }

    random_search_rows.append({
        "feature_set": best_feature_set,
        "model": model_name,
        "cv_brier": -search.best_score_,
        "train_brier": train_metrics["brier"],
        "test_brier": test_metrics["brier"],
        "val_brier": val_metrics["brier"],
        "train_auc": train_metrics["auc"],
        "test_auc": test_metrics["auc"],
        "val_auc": val_metrics["auc"],
        "gap_test_minus_train": test_metrics["brier"] - train_metrics["brier"],
        "gap_val_minus_train": val_metrics["brier"] - train_metrics["brier"]
    })

random_search_df = pd.DataFrame(random_search_rows).sort_values(
    ["test_brier", "val_brier", "cv_brier"],
    ascending=[True, True, True]
).reset_index(drop=True)

display(random_search_df)

RANDOM SEARCH: extratrees
Fitting 5 folds for each of 30 candidates, totalling 150 fits
RANDOM SEARCH: lightgbm
Fitting 5 folds for each of 30 candidates, totalling 150 fits
RANDOM SEARCH: xgboost
Fitting 5 folds for each of 30 candidates, totalling 150 fits
RANDOM SEARCH: catboost
Fitting 5 folds for each of 30 candidates, totalling 150 fits
RANDOM SEARCH: histgb
Fitting 5 folds for each of 30 candidates, totalling 150 fits


,feature_set,model,cv_brier,train_brier,test_brier,val_brier,train_auc,test_auc,val_auc,gap_test_minus_train,gap_val_minus_train
0,diff,extratrees,NaN,0.096984,0.198943,0.165022,0.985486,0.732919,0.854396,0.101959,0.068037
1,diff,xgboost,NaN,0.055621,0.230227,0.168395,0.997645,0.688406,0.842491,0.174606,0.112774
2,diff,catboost,NaN,0.000059,0.252128,0.151236,1.000000,0.712215,0.854396,0.252070,0.151178
3,diff,lightgbm,NaN,0.000012,0.316093,0.171164,1.000000,0.649068,0.852564,0.316080,0.171152
4,diff,histgb,NaN,0.000013,0.321149,0.190972,1.000000,0.627329,0.843407,0.321135,0.190958


In [33]:
for model_name, payload in random_search_results.items():
    print("=" * 80)
    print(model_name)
    print("BEST PARAMS:")
    print(payload["best_params"])
    print("CV BRIER :", payload["cv_brier"])
    print("TRAIN BRIER:", payload["train_metrics"]["brier"])
    print("TEST  BRIER:", payload["test_metrics"]["brier"])
    print("VAL   BRIER:", payload["val_metrics"]["brier"])

extratrees
BEST PARAMS:
{'criterion': 'log_loss', 'max_depth': 10, 'max_features': 0.9, 'min_samples_leaf': 8, 'min_samples_split': 8, 'n_estimators': 421}
CV BRIER : nan
TRAIN BRIER: 0.09698412817423246
TEST  BRIER: 0.19894335218646872
VAL   BRIER: 0.1650215633071177
lightgbm
BEST PARAMS:
{'colsample_bytree': np.float64(0.749816047538945), 'learning_rate': np.float64(0.06978211022792681), 'max_depth': 4, 'min_child_samples': 12, 'n_estimators': 1344, 'num_leaves': 117, 'reg_alpha': np.float64(0.06071989493441298), 'reg_lambda': np.float64(0.002511306167739001), 'subsample': np.float64(0.7836995567863468)}
CV BRIER : nan
TRAIN BRIER: 1.2473184595189604e-05
TEST  BRIER: 0.3160926261577159
VAL   BRIER: 0.17116404684938974
xgboost
BEST PARAMS:
{'colsample_bytree': np.float64(0.749816047538945), 'gamma': np.float64(2.0218499516556734), 'learning_rate': np.float64(0.038052091892039265), 'max_depth': 7, 'min_child_weight': 5, 'n_estimators': 421, 'reg_alpha': np.float64(0.004207053950287938)

In [34]:
top3_random = random_search_df.head(3)["model"].tolist()
top3_random

['extratrees', 'xgboost', 'catboost']

In [35]:
random_calibrated_results = {}
random_calibrated_rows = []

for model_name in top3_random:
    print("Kalibracja:", model_name)

    base_model = random_search_results[model_name]["best_model"]

    cal_model = CalibratedClassifierCV(
        estimator=base_model,
        method="sigmoid",
        cv=5
    )
    cal_model.fit(Xtr, y_train)

    train_proba = cal_model.predict_proba(Xtr)[:, 1]
    test_proba  = cal_model.predict_proba(Xte)[:, 1]
    val_proba   = cal_model.predict_proba(Xva)[:, 1]

    train_metrics = evaluate_proba(y_train, train_proba)
    test_metrics  = evaluate_proba(y_test, test_proba)
    val_metrics   = evaluate_proba(y_val, val_proba)

    cal_name = f"{model_name}_cal"

    random_calibrated_results[cal_name] = {
        "model": cal_model,
        "train_proba": train_proba,
        "test_proba": test_proba,
        "val_proba": val_proba,
        "train_metrics": train_metrics,
        "test_metrics": test_metrics,
        "val_metrics": val_metrics
    }

    random_calibrated_rows.append({
        "feature_set": best_feature_set,
        "model": cal_name,
        "train_brier": train_metrics["brier"],
        "test_brier": test_metrics["brier"],
        "val_brier": val_metrics["brier"],
        "train_auc": train_metrics["auc"],
        "test_auc": test_metrics["auc"],
        "val_auc": val_metrics["auc"],
        "gap_test_minus_train": test_metrics["brier"] - train_metrics["brier"],
        "gap_val_minus_train": val_metrics["brier"] - train_metrics["brier"]
    })

random_calibrated_df = pd.DataFrame(random_calibrated_rows).sort_values(
    ["test_brier", "val_brier"],
    ascending=[True, True]
).reset_index(drop=True)

display(random_calibrated_df)

Kalibracja: extratrees
Kalibracja: xgboost
Kalibracja: catboost


,feature_set,model,train_brier,test_brier,val_brier,train_auc,test_auc,val_auc,gap_test_minus_train,gap_val_minus_train
0,diff,extratrees_cal,0.105480,0.202166,0.162220,0.971274,0.731884,0.854396,0.096686,0.056741
1,diff,xgboost_cal,0.091712,0.216148,0.167437,0.993849,0.698758,0.851648,0.124436,0.075725
2,diff,catboost_cal,0.076529,0.217754,0.162704,1.000000,0.707039,0.863553,0.141224,0.086175


In [36]:
random_combined_df = pd.concat(
    [
        random_search_df.assign(source="base"),
        random_calibrated_df.assign(source="calibrated")
    ],
    ignore_index=True
).sort_values(
    ["test_brier", "val_brier"],
    ascending=[True, True]
).reset_index(drop=True)

display(random_combined_df)

,feature_set,model,cv_brier,train_brier,test_brier,val_brier,train_auc,test_auc,val_auc,gap_test_minus_train,gap_val_minus_train,source
0,diff,extratrees,NaN,0.096984,0.198943,0.165022,0.985486,0.732919,0.854396,0.101959,0.068037,base
1,diff,extratrees_cal,NaN,0.105480,0.202166,0.162220,0.971274,0.731884,0.854396,0.096686,0.056741,calibrated
2,diff,xgboost_cal,NaN,0.091712,0.216148,0.167437,0.993849,0.698758,0.851648,0.124436,0.075725,calibrated
3,diff,catboost_cal,NaN,0.076529,0.217754,0.162704,1.000000,0.707039,0.863553,0.141224,0.086175,calibrated
4,diff,xgboost,NaN,0.055621,0.230227,0.168395,0.997645,0.688406,0.842491,0.174606,0.112774,base
5,diff,catboost,NaN,0.000059,0.252128,0.151236,1.000000,0.712215,0.854396,0.252070,0.151178,base
6,diff,lightgbm,NaN,0.000012,0.316093,0.171164,1.000000,0.649068,0.852564,0.316080,0.171152,base
7,diff,histgb,NaN,0.000013,0.321149,0.190972,1.000000,0.627329,0.843407,0.321135,0.190958,base


In [37]:
random_pred_train = {}
random_pred_test = {}
random_pred_val = {}

for model_name, payload in random_search_results.items():
    random_pred_train[model_name] = payload["train_proba"]
    random_pred_test[model_name] = payload["test_proba"]
    random_pred_val[model_name] = payload["val_proba"]

for model_name, payload in random_calibrated_results.items():
    random_pred_train[model_name] = payload["train_proba"]
    random_pred_test[model_name] = payload["test_proba"]
    random_pred_val[model_name] = payload["val_proba"]

print(list(random_pred_test.keys()))

['extratrees', 'lightgbm', 'xgboost', 'catboost', 'histgb', 'extratrees_cal', 'xgboost_cal', 'catboost_cal']


In [38]:
top_k_random = 4
ensemble_random_members = random_combined_df.head(top_k_random).copy()

ensemble_random_members["weight"] = 1.0 / (ensemble_random_members["test_brier"] + 1e-9)
ensemble_random_members["weight"] = ensemble_random_members["weight"] / ensemble_random_members["weight"].sum()

display(ensemble_random_members[["model", "test_brier", "val_brier", "weight"]])

ensemble_random_train = np.zeros(len(y_train))
ensemble_random_test = np.zeros(len(y_test))
ensemble_random_val = np.zeros(len(y_val))

for _, row in ensemble_random_members.iterrows():
    model_name = row["model"]
    w = row["weight"]

    ensemble_random_train += w * random_pred_train[model_name]
    ensemble_random_test += w * random_pred_test[model_name]
    ensemble_random_val += w * random_pred_val[model_name]

ensemble_random_train_metrics = evaluate_proba(y_train, ensemble_random_train)
ensemble_random_test_metrics = evaluate_proba(y_test, ensemble_random_test)
ensemble_random_val_metrics = evaluate_proba(y_val, ensemble_random_val)

print("ENSEMBLE RANDOM SEARCH - TRAIN:")
print(ensemble_random_train_metrics)

print("\nENSEMBLE RANDOM SEARCH - TEST:")
print(ensemble_random_test_metrics)

print("\nENSEMBLE RANDOM SEARCH - VAL:")
print(ensemble_random_val_metrics)

,model,test_brier,val_brier,weight
0,extratrees,0.198943,0.165022,0.261912
1,extratrees_cal,0.202166,0.162220,0.257737
2,xgboost_cal,0.216148,0.167437,0.241064
3,catboost_cal,0.217754,0.162704,0.239287


ENSEMBLE RANDOM SEARCH - TRAIN:
{'brier': 0.09036747607751801, 'logloss': 0.3413601502373384, 'auc': 0.9969870310328857, 'accuracy': 0.9718631178707224, 'f1': 0.9727740986019132, 'precision': 0.9734904270986745, 'recall': 0.9720588235294118, 'pred_mean': 0.5147463719495837}

ENSEMBLE RANDOM SEARCH - TEST:
{'brier': 0.2066436479572061, 'logloss': 0.5972534557227693, 'auc': 0.7122153209109731, 'accuracy': 0.6865671641791045, 'f1': 0.5714285714285714, 'precision': 0.5, 'recall': 0.6666666666666666, 'pred_mean': 0.43340870806508663}

ENSEMBLE RANDOM SEARCH - VAL:
{'brier': 0.1623931057614117, 'logloss': 0.49901918403158907, 'auc': 0.8598901098901099, 'accuracy': 0.7910447761194029, 'f1': 0.8205128205128205, 'precision': 0.8205128205128205, 'recall': 0.8205128205128205, 'pred_mean': 0.5116256067223873}


In [39]:
comparison_rows = []

for _, row in random_combined_df.iterrows():
    comparison_rows.append({
        "kind": "single_model",
        "model": row["model"],
        "test_brier": row["test_brier"],
        "val_brier": row["val_brier"],
        "test_auc": row["test_auc"],
        "val_auc": row["val_auc"]
    })

comparison_rows.append({
    "kind": "ensemble",
    "model": "ensemble_random_top4",
    "test_brier": ensemble_random_test_metrics["brier"],
    "val_brier": ensemble_random_val_metrics["brier"],
    "test_auc": ensemble_random_test_metrics["auc"],
    "val_auc": ensemble_random_val_metrics["auc"]
})

random_final_df = pd.DataFrame(comparison_rows).sort_values(
    ["val_brier", "test_brier"],
    ascending=[True, True]
).reset_index(drop=True)

display(random_final_df)

,kind,model,test_brier,val_brier,test_auc,val_auc
0,single_model,catboost,0.252128,0.151236,0.712215,0.854396
1,single_model,extratrees_cal,0.202166,0.162220,0.731884,0.854396
2,ensemble,ensemble_random_top4,0.206644,0.162393,0.712215,0.859890
3,single_model,catboost_cal,0.217754,0.162704,0.707039,0.863553
4,single_model,extratrees,0.198943,0.165022,0.732919,0.854396
5,single_model,xgboost_cal,0.216148,0.167437,0.698758,0.851648
6,single_model,xgboost,0.230227,0.168395,0.688406,0.842491
7,single_model,lightgbm,0.316093,0.171164,0.649068,0.852564
8,single_model,histgb,0.321149,0.190972,0.627329,0.843407


In [40]:
top5 = random_combined_df.sort_values(["test_brier", "val_brier"]).head(5)

for i, row in top5.iterrows():
    model_name = row["model"]
    print("=" * 100)
    print(f"MODEL: {model_name}")
    print(f"test_brier = {row['test_brier']:.6f}, val_brier = {row['val_brier']:.6f}")

    if model_name in random_search_results:
        print("PARAMS:")
        print(random_search_results[model_name]["best_params"])
    elif model_name in random_calibrated_results:
        base_name = model_name.replace("_cal", "")
        print("PARAMS:")
        print(random_search_results[base_name]["best_params"])
        print("CALIBRATION: sigmoid, cv=3")

MODEL: extratrees
test_brier = 0.198943, val_brier = 0.165022
PARAMS:
{'criterion': 'log_loss', 'max_depth': 10, 'max_features': 0.9, 'min_samples_leaf': 8, 'min_samples_split': 8, 'n_estimators': 421}
MODEL: extratrees_cal
test_brier = 0.202166, val_brier = 0.162220
PARAMS:
{'criterion': 'log_loss', 'max_depth': 10, 'max_features': 0.9, 'min_samples_leaf': 8, 'min_samples_split': 8, 'n_estimators': 421}
CALIBRATION: sigmoid, cv=3
MODEL: xgboost_cal
test_brier = 0.216148, val_brier = 0.167437
PARAMS:
{'colsample_bytree': np.float64(0.749816047538945), 'gamma': np.float64(2.0218499516556734), 'learning_rate': np.float64(0.038052091892039265), 'max_depth': 7, 'min_child_weight': 5, 'n_estimators': 421, 'reg_alpha': np.float64(0.004207053950287938), 'reg_lambda': np.float64(0.0017073967431528124), 'subsample': np.float64(0.9464704583099741)}
CALIBRATION: sigmoid, cv=3
MODEL: catboost_cal
test_brier = 0.217754, val_brier = 0.162704
PARAMS:
{'depth': 7, 'iterations': 1160, 'l2_leaf_reg': np